# Phase 2B: 2026 외부 시즌 검증 (원 모델 가중치 고정)

**2025년에 학습된 Pre-game C1 및 In-game DNN을 2026 정규시즌에 고정 적용. 2026년 9월 27일 BAL@NYY 경기가 취소되어 완료 표본은 2,429경기입니다.**

업로드: `phase1c2_output.zip`, `phase1d_output.zip`. 이 노트북은 외부 시즌의 완료 경기를 **사후 수집하여 재구성**하므로 *retrospective external-season evaluation*, 당시 실시간으로 기록한 *prospective forecasts*가 아닙니다. 2026 과거 일정 API의 `probablePitcher` 정보가 실제 경기 전 공개정보와 일치한다는 보장이 없으므로 Pre-game 평가는 잠정적이며 이 한계를 명시해야 합니다. 미래 기록을 팀/투수 통계에 반영하지 않도록 날짜 이전 누적값만 사용합니다.

API 호출량이 많습니다. 중간 종료에 대비하여 In-game 피드는 Colab/Drive 캐시를 재사용합니다. 결과 CSV와 오류 로그를 포함한 ZIP을 다운로드합니다.

In [ ]:
!pip -q install pandas numpy requests tqdm scikit-learn
import pandas as pd,numpy as np,requests,time,json,zipfile,io,os
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score,roc_auc_score,brier_score_loss,log_loss
from google.colab import files
YEAR=2026
OUT=Path('/content/mlb_phase2b_2026');OUT.mkdir(exist_ok=True)
BASE='https://statsapi.mlb.com/api/v1';s=requests.Session();s.headers.update({'User-Agent':'academic research season generalization 2026','Accept':'application/json'})
def get(path,params=None,retries=5):
    for k in range(retries):
        try:
            r=s.get(BASE+path,params=params,timeout=(15,45))
            if r.status_code in (429,500,502,503,504):raise RuntimeError(f'HTTP {r.status_code}')
            r.raise_for_status();return r.json()
        except Exception:
            if k==retries-1:raise
            time.sleep(min(2**k+1,20))
u=files.upload();assert {'phase1c2_output.zip','phase1d_output.zip'}.issubset(u),'2개 zip 파일 필요'
def exported(z,f):return json.loads(zipfile.ZipFile(io.BytesIO(u[z])).read(f))
pre=exported('phase1c2_output.zip','selected_pregame_export.json'); ing=exported('phase1d_output.zip','ingame_model_export.json')
original_audit=json.loads(zipfile.ZipFile(io.BytesIO(u['phase1d_output.zip'])).read('ingame_audit.json'))
print('Pre-game features:',len(pre['features']),'In-game:',len(ing['features']))

## 1. 2026 완료 정규시즌 일정 및 정확한 경기 수 감사

In [ ]:
all_raw=[]
for m in tqdm(pd.period_range('2026-03','2026-10',freq='M'),desc='2026 schedule'):
    v=get('/schedule',{'sportId':1,'startDate':str(m.start_time.date()),'endDate':str(m.end_time.date()),'hydrate':'probablePitcher'})
    for day in v.get('dates',[]):all_raw.extend(day.get('games',[]))
    time.sleep(.10)
byid={}
for g in all_raw:
    if g.get('gameType')!='R':continue
    state=g.get('status',{}).get('codedGameState')
    if state!='F':continue
    home=g.get('teams',{}).get('home',{});away=g.get('teams',{}).get('away',{})
    if home.get('score') is None or away.get('score') is None:continue
    gid=int(g['gamePk']); byid[gid]=g
rows=[]
for gid,g in byid.items():
    h=g['teams']['home'];a=g['teams']['away']
    rows.append({'gamePk':gid,'date':str(g.get('officialDate') or g['gameDate'][:10]),'home_id':int(h['team']['id']),'away_id':int(a['team']['id']),
                 'home_score':int(h['score']),'away_score':int(a['score']),
                 'home_pitcher_id':(h.get('probablePitcher') or {}).get('id'),'away_pitcher_id':(a.get('probablePitcher') or {}).get('id')})
games=pd.DataFrame(rows).sort_values(['date','gamePk']).reset_index(drop=True)
assert not games.empty and not (games.home_score==games.away_score).any()
games['target']=(games.home_score>games.away_score).astype(int)
print('2026 정규시즌 완료 경기',len(games),'고유 게임',games.gamePk.nunique())
# 2026-09-27 Baltimore Orioles @ New York Yankees 정규시즌 최종전은 취소되었고 재편성되지 않았습니다.
# MLB 공식 발표: https://www.mlb.com/news/orioles-yankees-september-27-game-canceled-due-to-weather
# 따라서 2026 완료된 정규시즌의 기대 표본은 2,429경기입니다.
regular_by_id={int(g['gamePk']):g for g in all_raw if g.get('gameType')=='R'}
not_final=[]
for gid,g in regular_by_id.items():
    if gid not in byid:
        status=g.get('status') or {}
        not_final.append({'gamePk':gid,'officialDate':g.get('officialDate'),
            'away':(g.get('teams',{}).get('away',{}).get('team') or {}).get('name'),
            'home':(g.get('teams',{}).get('home',{}).get('team') or {}).get('name'),
            'status':status.get('detailedState'),'codedGameState':status.get('codedGameState')})
not_final_df=pd.DataFrame(not_final)
not_final_df.to_csv(OUT/'2026_regular_season_not_completed.csv',index=False)
print('일정상 정규시즌 게임:',len(regular_by_id),'완료:',len(games),'미완료·취소:',len(not_final_df))
if not not_final_df.empty: display(not_final_df)
assert len(games)==2429 and games.gamePk.nunique()==2429, (
    f'예상 완료 경기 2429, 실제 {len(games)}: 일정 API 조회 및 취소 경기 상태 확인 필요')
assert len(not_final_df)==1, '2026 미완료 정규시즌 경기 수 예상(1)과 다릅니다. 일정 확인 필요.'
missing=not_final_df.iloc[0]
assert str(missing['officialDate'])=='2026-09-27' and {missing['home'],missing['away']}=={'New York Yankees','Baltimore Orioles'}, (
    f'미완료 경기 식별 불일치: {missing.to_dict()}')
print('취소된 BAL @ NYY 경기 확인: 분석 제외, 2429 완료 경기만 사용')
games.to_csv(OUT/'2026_games.csv',index=False)

## 2. 날짜 이전 팀 성적 — 2025년과 동일 정책

In [ ]:
t=defaultdict(lambda:{'gp':0,'w':0,'rs':0,'ra':0,'results':[]})
team_rows=[]
for d,day in games.groupby('date',sort=True):
    # 단계 1: 하루의 모든 경기 특성을 당일 업데이트 전에 계산
    for g in day.itertuples(index=False):
        r={'gamePk':g.gamePk,'date':d,'target':g.target,'home_id':g.home_id,'away_id':g.away_id,
           'home_pitcher_id':g.home_pitcher_id,'away_pitcher_id':g.away_pitcher_id}
        for side,tid in [('home',g.home_id),('away',g.away_id)]:
            v=t[tid];gp=v['gp']
            r[f'{side}_prior_games']=gp
            r[f'{side}_win_pct']=v['w']/gp if gp>=10 else .5
            r[f'{side}_winpct_imputed']=(gp<10)
            r[f'{side}_runs_per_game']=v['rs']/gp if gp else 4.5
            r[f'{side}_ra_per_game']=v['ra']/gp if gp else 4.5
            r[f'{side}_run_diff']=(v['rs']-v['ra'])/gp if gp else 0.
            last=v['results'][-10:];r[f'{side}_recent_form']=float(np.mean(last)) if last else .5
        team_rows.append(r)
    # 단계 2: 그 날짜 모든 경기 결과를 다음 날짜에 반영
    for g in day.itertuples(index=False):
        for side,tid,rs,ra,w in [('home',g.home_id,g.home_score,g.away_score,g.target),('away',g.away_id,g.away_score,g.home_score,1-g.target)]:
            v=t[tid];v['gp']+=1;v['w']+=int(w);v['rs']+=int(rs);v['ra']+=int(ra);v['results'].append(int(w))
team=pd.DataFrame(team_rows)
assert len(team)==len(games) and team.gamePk.nunique()==len(games)
team_counts=pd.DataFrame([{'team_id':tid,'completed_games':v['gp']} for tid,v in sorted(t.items())])
team_counts.to_csv(OUT/'2026_team_game_counts.csv',index=False)
print('팀별 완료 경기 수:',team_counts.completed_games.value_counts().to_dict())
assert len(team_counts)==30 and sorted(team_counts.completed_games.value_counts().to_dict().items())==[(161,2),(162,28)], (
    '팀별 경기 수 예상(161경기 2팀, 162경기 28팀)과 다릅니다.')
assert team_counts.completed_games.sum()==len(games)*2
canceled_ids={int((g.get('teams',{}).get(side,{}).get('team') or {}).get('id')) for g in regular_by_id.values()
              if int(g['gamePk'])==int(missing['gamePk']) for side in ('home','away')}
assert set(team_counts.loc[team_counts.completed_games==161,'team_id'])==canceled_ids, '161경기 팀이 취소 경기 양 팀과 일치하지 않습니다.' 
team.to_csv(OUT/'2026_team_features.csv',index=False)
print('팀 28개 162경기, 취소 경기 양 팀 161경기 확인 / 10경기 미만 승률 대체 게임 수:',int((team.home_winpct_imputed|team.away_winpct_imputed).sum()))
display(team.head(3))

## 3. 2026 투수 시즌 로그 수집 (중간 결과 저장)

과거 probablePitcher 데이터는 실제 발표 시점의 기록이 아닐 수 있습니다. 따라서 본 Pre-game 평가에는 정보 가용성의 한계가 존재합니다.

In [ ]:
def ip_to_outs(v):
    # MLB IP 표기 5.2 = 5이닝 2아웃; 소수점 계산 금지
    st=str(v or '0.0'); a,b=(st.split('.')+['0'])[:2]
    if b not in ('0','1','2'): raise ValueError(f'IP 표기 이상: {v}')
    return int(a)*3+int(b)
assert ip_to_outs('5.2')==17 and ip_to_outs('6.1')==19
ids=pd.to_numeric(pd.concat([games.home_pitcher_id,games.away_pitcher_id]),errors='coerce').dropna().astype(int).unique()
print('조회할 고유 선발 예정 투수:',len(ids))
logs=[]; problems=[]
for pid in tqdm(ids,desc='pitcher logs'):
    try:
        data=get(f'/people/{int(pid)}/stats',{'stats':'gameLog','group':'pitching','season':YEAR,'gameType':'R'})
        blocks=data.get('stats',[])
        splits=[x for b in blocks for x in b.get('splits',[])]
        for z in splits:
            dt=z.get('date') or z.get('gameDate')
            if not dt: continue
            st=z.get('stat',{})
            logs.append({'pitcher_id':int(pid),'date':str(dt)[:10],
                         'gamePk':(z.get('game') or {}).get('gamePk'),
                         'outs':ip_to_outs(st.get('inningsPitched','0.0')),
                         'er':int(st.get('earnedRuns',0) or 0),
                         'h':int(st.get('hits',0) or 0),
                         'bb':int(st.get('baseOnBalls',0) or 0),
                         'so':int(st.get('strikeOuts',0) or 0)})
    except Exception as e:
        problems.append({'pitcher_id':int(pid),'error':str(e)})
    time.sleep(.05)
logs=pd.DataFrame(logs,columns=['pitcher_id','date','gamePk','outs','er','h','bb','so'])
if not logs.empty:
    logs=logs.drop_duplicates(['pitcher_id','gamePk'],keep='first') if logs.gamePk.notna().all() else logs.drop_duplicates(['pitcher_id','date','gamePk'],keep='first')
    logs.sort_values(['pitcher_id','date'],inplace=True)
logs.to_csv(OUT/'2026_pitcher_gamelogs.csv',index=False)
pd.DataFrame(problems,columns=['pitcher_id','error']).to_csv(OUT/'pitcher_api_errors.csv',index=False)
print('게임로그',len(logs),'행 / API 오류',len(problems))
if len(problems):print('API 오류 예시',problems[:5])
if logs.empty: raise RuntimeError('투수 게임로그를 전혀 수집하지 못했습니다. ZIP 대신 오류 메시지를 전달해 주세요.')
display(logs.head(5))

## 4. 2025 학습 모델의 중앙값 대체 + 2026 Pre-game 예측

In [ ]:
# 날짜 이전만 사용하는 누적 기록 (오늘 기록은 포함되지 않음)
neutral={'era':4.30,'whip':1.30,'k9':8.50}
by_pid={int(k):v.sort_values('date') for k,v in logs.groupby('pitcher_id')}
lookup={}
for g in games.itertuples(index=False):
    for side in ('home','away'):
        pid=getattr(g,f'{side}_pitcher_id')
        if pd.isna(pid):
            lookup[(g.gamePk,side)]=dict(neutral,source='missing_pitcher_id',prior_outs=0);continue
        prior=by_pid.get(int(pid))
        if prior is None:
            lookup[(g.gamePk,side)]=dict(neutral,source='no_gamelog',prior_outs=0);continue
        v=prior.loc[prior.date<g.date,['outs','er','h','bb','so']].sum()
        outs=int(v['outs'])
        if outs==0:
            lookup[(g.gamePk,side)]=dict(neutral,source='no_prior_outs',prior_outs=0);continue
        ip=outs/3
        lookup[(g.gamePk,side)]={'era':9*v['er']/ip,'whip':(v['h']+v['bb'])/ip,'k9':9*v['so']/ip,
                                  'source':'prior_game_logs','prior_outs':outs}
feat=team.copy()
for side in ('home','away'):
    for key in ['era','whip','k9','source','prior_outs']:
        feat[f'{side}_sp_{key}']=[lookup[(int(gid),side)][key] for gid in feat.gamePk]
feat['win_pct_diff']=feat.home_win_pct-feat.away_win_pct
feat['run_diff_diff']=feat.home_run_diff-feat.away_run_diff
feat['form_diff']=feat.home_recent_form-feat.away_recent_form
feat['sp_era_diff']=feat.away_sp_era-feat.home_sp_era # 기존 코드의 방향 유지
feat['sp_whip_diff']=feat.away_sp_whip-feat.home_sp_whip
feat['sp_k9_diff']=feat.home_sp_k9-feat.away_sp_k9
# home_advantage는 모든 표본에서 1인 상수이므로 분석 입력에서 제외 (22개)
features=['home_win_pct','away_win_pct','win_pct_diff','home_run_diff','away_run_diff','run_diff_diff',
          'home_runs_per_game','away_runs_per_game','home_ra_per_game','away_ra_per_game',
          'home_recent_form','away_recent_form','form_diff',
          'home_sp_era','away_sp_era','sp_era_diff','home_sp_whip','away_sp_whip','sp_whip_diff',
          'home_sp_k9','away_sp_k9','sp_k9_diff']
assert len(features)==22 and feat[features].notna().all().all()
feat.to_csv(OUT/'2026_pregame_features.csv',index=False)
print(f'{len(feat):,}경기 특성 계산 완료 (현재는 진단용, 재학습 전 대체값 재검토 필요)')
print('선발 기록 출처:')
for side in ('home','away'):print(side,feat[f'{side}_sp_source'].value_counts().to_dict())
display(feat[['gamePk','date','home_prior_games','away_prior_games','home_sp_source','away_sp_source']].head(8))

In [ ]:
# 2025 학습 때 얻은 임퓨테이션 중앙값을 그대로 사용
med={'era':3.8129770992366416,'whip':1.2608695652173914,'k9':8.052631578947368}
for side in ('home','away'):
    for k in ('era','whip','k9'):
        feat.loc[feat[f'{side}_sp_source']!='prior_game_logs',f'{side}_sp_{k}']=med[k]
feat['sp_era_diff']=feat.away_sp_era-feat.home_sp_era
feat['sp_whip_diff']=feat.away_sp_whip-feat.home_sp_whip
feat['sp_k9_diff']=feat.home_sp_k9-feat.away_sp_k9
assert set(pre['features'])==set(features) and feat[pre['features']].notna().all().all()
def forward(frame,model):
    a=(frame[model['features']].to_numpy(dtype=np.float64)-np.array(model['scaler']['mean']))/np.array(model['scaler']['scale'])
    for layer in model['layers']:
        a=a@np.array(layer['W'])+np.array(layer['b'])
        a=1/(1+np.exp(-np.clip(a,-40,40))) if a.shape[1]==1 else np.maximum(a,0)
    return a[:,0]
def getmetrics(y,p):
    return {'n':len(y),'accuracy':accuracy_score(y,p>=.5),'auc':roc_auc_score(y,p),'brier':brier_score_loss(y,p),'log_loss':log_loss(y,np.clip(p,1e-8,1-1e-8))}
feat['prob_pregame_dnn']=forward(feat,pre)
feat.to_csv(OUT/'2026_pregame_predictions.csv',index=False)
print(getmetrics(feat.target,feat.prob_pregame_dnn))
print('2026 probablePitcher missing',feat[['home_pitcher_id','away_pitcher_id']].isna().sum().to_dict())

## 5. 2026 In-game 반이닝 시작 전 관측치 수집

Phase 1D의 동일한 `prepare_game` 보완 함수를 재사용합니다. 캐시를 Drive에 저장하려면 USE_DRIVE=True.

In [ ]:
USE_DRIVE=False  # 런타임 재시작 후 캐시 보존은 True와 구글 드라이브 연결 권장
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_BASE=Path('/content/drive/MyDrive/MLB_2026_External')
else:
    CACHE_BASE=Path('/content/MLB_2026_External')
CACHE_BASE.mkdir(parents=True,exist_ok=True)
CACHE=CACHE_BASE/'2026_ingame_cache.jsonl'
ERROR_LOG=CACHE_BASE/'2026_ingame_api_errors.jsonl'
print('2026 cache:',CACHE)

In [ ]:
SESSION=requests.Session()
SESSION.headers.update({'User-Agent':'academic-research-mlb-phase2b/1.0','Accept':'application/json'})
def get_feed(gid,retries=5):
    url=f'https://statsapi.mlb.com/api/v1.1/game/{int(gid)}/feed/live'
    last=None
    for trial in range(retries):
        try:
            r=SESSION.get(url,timeout=(15,45))
            if r.status_code in (429,500,502,503,504):
                time.sleep(min(2**trial+1,20));continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last=repr(e); time.sleep(min(2**trial+1,20))
    raise RuntimeError(f'gamePk {gid}: API 실패 {last}')

def prepare_game(gid,feed):
    games=feed.get('gameData',{}).get('teams',{})
    live=feed.get('liveData',{})
    score=live.get('linescore',{})
    innings=score.get('innings') or []
    allplays=live.get('plays',{}).get('allPlays') or []
    if not innings or not allplays: raise ValueError('missing innings or allPlays')
    # 실제 플레이가 한 번이라도 기록된 반이닝만 생성
    appeared=set()
    for p in allplays:
        a=p.get('about') or {}
        num=a.get('inning'); half=str(a.get('halfInning','')).lower()
        if num and half in ('top','bottom'): appeared.add((int(num),half))
    if not appeared: raise ValueError('no started half-innings')
    # API half-inning summary가 비어 있는 예외 경기에서 allPlays로 보완한다.
    # 점수와 안타 합계가 최종 linescore와 맞지 않으면 해당 경기는 통과시키지 않는다.
    derived={}
    hit_events={'single','double','triple','home_run'}
    for play in allplays:
        about=play.get('about') or {}
        num=about.get('inning');half=str(about.get('halfInning','')).lower()
        if not num or half not in ('top','bottom'):continue
        key=(int(num),half)
        state=derived.setdefault(key,{'runs':0,'hits':0})
        event=(play.get('result') or {}).get('eventType')
        if event in hit_events:state['hits']+=1
        for runner in play.get('runners') or []:
            details=runner.get('details') or {}
            movement=runner.get('movement') or {}
            if details.get('isScoring') is True or movement.get('end')=='score':
                state['runs']+=1
    fallback_halves=[]
    completed={}
    for inn in innings:
        n=inn.get('num')
        if n is None:continue
        for half, side in [('top','away'),('bottom','home')]:
            data=inn.get(side) or {}
            if (int(n),half) not in appeared:continue
            runs=data.get('runs');hits=data.get('hits')
            if runs is None or hits is None:
                fallback=derived.get((int(n),half))
                if fallback is None:raise ValueError(f'missing runs/hits and play data {n} {half}')
                fallback_halves.append({'inning':int(n),'half':half,'missing_runs':runs is None,'missing_hits':hits is None})
                if runs is None:runs=fallback['runs']
                if hits is None:hits=fallback['hits']
            completed[(int(n),half)]={'runs':int(runs),'hits':int(hits)}
    missing=appeared-set(completed)
    if missing:raise ValueError(f'missing half summary: {sorted(missing)[:3]}')
    seq=sorted(appeared,key=lambda x:(x[0],0 if x[1]=='top' else 1))
    hr=ar=hh=ah=0;rows=[]
    # 스냅샷은 각 반이닝의 첫 투구 전 시점. 마지막 반이닝 종료 상태는 절대 만들지 않음.
    for n,half in seq:
        rows.append({'gamePk':int(gid),'inning':n,'is_bottom':int(half=='bottom'),
                     'score_diff':hr-ar,'home_runs':hr,'away_runs':ar,
                     'home_hits':hh,'away_hits':ah,'abs_score_diff':abs(hr-ar),
                     'snapshot':'pre_half_inning'})
        v=completed[(n,half)]
        if half=='top': ar+=v['runs'];ah+=v['hits']
        else: hr+=v['runs'];hh+=v['hits']
    # 검증: 모든 반이닝을 합산한 점수가 linescore의 최종 점수와 일치해야 함
    finalh=(score.get('teams') or {}).get('home',{}).get('runs')
    finala=(score.get('teams') or {}).get('away',{}).get('runs')
    if finalh is None or finala is None: raise ValueError('missing final runs')
    if (hr,ar)!=(int(finalh),int(finala)):
        raise ValueError(f'run checksum mismatch: summed {(hr,ar)} vs final {(finalh,finala)}')
    if int(finalh)==int(finala):raise ValueError('tie final score')
    if fallback_halves:
        finalhh=(score.get('teams') or {}).get('home',{}).get('hits')
        finalah=(score.get('teams') or {}).get('away',{}).get('hits')
        if finalhh is None or finalah is None:
            raise ValueError('fallback hit checksum unavailable')
        if (hh,ah)!=(int(finalhh),int(finalah)):
            raise ValueError(f'fallback hit checksum mismatch: {(hh,ah)} vs {(finalhh,finalah)}')
    return {'gamePk':int(gid),'fallback_halves':fallback_halves,
            'home_final':int(finalh),'away_final':int(finala),
            'home_win':int(int(finalh)>int(finala)),'snapshots':rows,
            'n_started_halves':len(seq),'n_extras':sum(n>9 for n,_ in seq)}



In [ ]:
cache={};bad=[]
if CACHE.exists():
    for line in CACHE.open():
        try:
            d=json.loads(line);cache[int(d['gamePk'])]=d
        except Exception as e:bad.append(str(e))
print('cache',len(cache),'invalid lines',len(bad))
errors=[];targets=dict(zip(games.gamePk,games.target))
for idx,gid in enumerate(tqdm(games.gamePk,desc='2026 game feeds')):
    gid=int(gid)
    if gid in cache:continue
    try:
        obj=prepare_game(gid,get_feed(gid))
        if int(obj['home_win'])!=int(targets[gid]):raise ValueError('target mismatch')
        cache[gid]=obj
        with CACHE.open('a') as f:f.write(json.dumps(obj,ensure_ascii=False)+'\n')
    except Exception as e:
        errors.append({'gamePk':gid,'error':str(e)})
        with ERROR_LOG.open('a') as f:f.write(json.dumps(errors[-1],ensure_ascii=False)+'\n')
    if idx%100==0 and idx>0:print(idx,len(cache),'errors',len(errors))
print('cached',len(cache),'errors',len(errors))
if errors:display(pd.DataFrame(errors).head(20))
assert len(cache)==len(games),'누락 경기 복구 전 분석하지 마세요.'
fr=[]
for g in games.itertuples(index=False):
    for r in cache[int(g.gamePk)]['snapshots']:
        fr.append({**r,'date':g.date,'target':int(g.target)})
ingame_df=pd.DataFrame(fr).sort_values(['date','gamePk','inning','is_bottom']).reset_index(drop=True)
assert ingame_df.gamePk.nunique()==len(games) and ingame_df[ing['features']].notna().all().all()
ingame_df['prob_ingame_dnn']=forward(ingame_df,ing)
ingame_df.to_csv(OUT/'2026_ingame_predictions.csv',index=False)
print('2026 In-game',getmetrics(ingame_df.target,ingame_df.prob_ingame_dnn))
print('Inning table')
by=[]
for inning,g in ingame_df.groupby('inning'):
    row={'inning':int(inning),**getmetrics(g.target,g.prob_ingame_dnn)};by.append(row)
pd.DataFrame(by).to_csv(OUT/'2026_ingame_by_inning.csv',index=False)
display(pd.DataFrame(by))

## 6. API 요청 지연 측정 및 결과 ZIP

단위 요청 시간은 네트워크·MLB 서버 상태의 영향을 받습니다. 추론시간과 구별해서 보고합니다. 동일 날짜의 2025/2026 일반화 성능 비교 시 표본과 관측시점 조건 차이를 기술해야 합니다.

In [ ]:
lat=[]
for k in range(20):
    gid=int(games.iloc[k].gamePk)
    for kind,path,params in [('schedule','/schedule',{'sportId':1,'date':games.iloc[k].date}),('live_feed',None,None)]:
        t0=time.perf_counter()
        try:
            if kind=='schedule':get(path,params)
            else:get_feed(gid)
            status='ok'
        except Exception as e:status=repr(e)
        lat.append({'kind':kind,'gamePk':gid,'elapsed_ms':(time.perf_counter()-t0)*1000,'status':status})
pd.DataFrame(lat).to_csv(OUT/'2026_api_latency_ms.csv',index=False)
print(pd.DataFrame(lat).groupby('kind').elapsed_ms.describe())
audit={'year':YEAR,'games':len(games),'pregame_probable_pitcher_preannouncement_verified':False,
       'external_validation_type':'retrospective external-season; not logged prospective predictions',
       'pre_model':'fixed phase1c2 C1','in_model':'fixed phase1d DNN',
       'in_snapshots':len(ingame_df),'feed_errors':errors,'latency_trials':len(lat),
       'scheduled_regular_season_games':len(regular_by_id),
       'cancelled_games':not_final,
       'team_completed_game_counts':team_counts.to_dict(orient='records')}
(OUT/'2026_external_audit.json').write_text(json.dumps(audit,indent=2,ensure_ascii=False))
name='phase2b_2026_external_output.zip'
with zipfile.ZipFile(name,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        if f.is_file():z.write(f,arcname=f.name)
print('ZIP',name,Path(name).stat().st_size)
files.download(name)